In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import IPython

from src.dataset import AudioDataset

/home/jp/miniconda3/envs/mgr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Torch compile

In [2]:
import torch
from src.loss import MultiScaleSTFTLoss

torch._logging.set_logs(graph_code=True)

In [3]:
# Returns the result of running `fn()` and the time it took for `fn()` to run,
# in seconds. We use CUDA events and synchronization for the most accurate
# measurements.
def timed(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, start.elapsed_time(end) / 1000

In [ ]:
cuda = torch.device("cuda:0")
loss_compiled = torch.compile(MultiScaleSTFTLoss([2048, 1024, 512, 256, 128]), backend="cudagraphs").to(cuda)
loss_raw = MultiScaleSTFTLoss([2048, 1024, 512, 256, 128]).to(cuda)

x, y = torch.randn((10, 2, 2000), device=cuda), torch.randn((10, 2, 2000), device=cuda)
loss_compiled(x, y)

V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code] TRACED GRAPH
V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code]  ===== pre insert_deferred_runtime_asserts __compiled_fn_51 =====
V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code]  <eval_with_key>.7 class GraphModule(torch.nn.Module):
V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code]     def forward(self, L_x_: "f32[10, 2, 2000]"):
V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code]         l_x_ = L_x_
V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code]         
V0228 23:07:42.213000 3858996 site-packages/torch/fx/passes/runtime_assert.py:118] [1/1_1] [__graph_code]          # File: /home/jp/mgr2024/src/loss.py:

tensor(6.0724, device='cuda:0')

In [17]:
from timeit import timeit

timeit("loss_compiled(x, y)", globals=locals(), number=5000)

56.82223436515778

In [18]:
timeit("loss_raw(x, y)", globals=locals(), number=5000)

21.28980154497549

In [18]:
print("compile:", timed(lambda: loss_compiled(x, y))[1])
print("eager:", timed(lambda: loss_raw(x, y))[1])

compile: 0.019259424209594728
eager: 0.014740511894226075
